In [2]:
import itertools
import pandas as pd
import geopandas as gpd
from pathlib import Path
from libpysal import graph
from esda.moran import Moran
from joblib import Parallel, delayed
import numpy as np
import re

In [3]:
shapes = ['square','pent','hex']
sizes = [25,100,400]
corruption_methods = ['random','borders','local']
n_runs = 10
c_runs = 10
rhos = np.arange(-0.9, 1.0, 0.1)
GRAPH_NAME_RE = re.compile(r"g_missing(\d+)_run(\d+)\.parquet$") 

In [4]:
def get_available_intensities(shape, size, method, c_run):
    # get number of missig edges/corruption intensities for specific size/shape/method
    folder = Path(f"graphs/{shape}/size_{size}/no_isolates/{method}")
    pattern = re.compile(rf"g_missing(\d+)_run{c_run}\.parquet$")
    return sorted(
        int(pattern.match(f.name).group(1))
        for f in folder.glob(f"g_missing*_run{c_run}.parquet")
        if pattern.match(f.name)
    )

def parse_missing_edges(graph_path):
    # Derive the missing-edge count from the graph filename
    m = GRAPH_NAME_RE.search(str(graph_path))
    if not m:
        raise ValueError(f"could not parse missing-edge count from: {graph_path}")
    return int(m.group(1))
 
 
def select_sample_graphs(available_intensities, n_sample=20):
    """Linearly sample intensities from what's actually available.
    Always keeps the most-corrupted (max) graph."""
    available_intensities = sorted(available_intensities)
    max_idx = len(available_intensities) - 1
 
    idxs = np.linspace(0, max_idx, n_sample, dtype=int)
    idxs = idxs[1:]  # drop the least-corrupted graph
    idxs = sorted(set(idxs))
 
    return [available_intensities[i] for i in idxs]

In [5]:
def _process_graph(shape, size, method, c_run, intensity, gdf, n_runs, rhos):
    graph_path = (
        f"graphs/{shape}/size_{size}/no_isolates/{method}/"
        f"g_missing{intensity}_run{c_run}.parquet"
    )
    missing_edges = parse_missing_edges(graph_path) # get number of missing edges

    g = graph.read_parquet(graph_path)
    wm = g.to_W()
    wm.transform = 'r'
    id_order = list(wm.id_order)
    gdf_local = gdf.iloc[id_order]
    global_rows = []

    # global I computation loop
    for n_run in range(n_runs):
        for rho in rhos:
            rho_clean = rho + 0.0  # -0.0 -> 0.0
            col = f"rho_{rho_clean:.1f}_run_{n_run}"
            y = gdf_local[col].values

            mi = Moran(y, wm)
            global_rows.append({
                "shape": shape, "size": size, "corruption_method": method,
                "rho": rho, "missing_edges": missing_edges,
                "corruption_run": c_run, "data_run": n_run,
                "moran_i": mi.I, "moran_p": mi.p_sim, "moran_z": mi.z_sim,
            })

    return global_rows

In [7]:
methods_by_size = {
    25: ['random'],  # size 25 only has random removal
}

N_SAMPLE = 20    # sampled intensities
N_JOBS = -1

for shape, size in itertools.product(shapes, sizes):
    gdf = gpd.read_parquet(f"data/autocorrelation/gdf_{shape}_{size}.parquet").reset_index(drop=True)
    methods_for_this_size = methods_by_size.get(size, corruption_methods)

    for method in methods_for_this_size:
        jobs = [
            (c_run, intensity)
            for c_run in range(c_runs)
            for intensity in select_sample_graphs(
                get_available_intensities(shape, size, method, c_run), n_sample=N_SAMPLE
            )
        ]

        global_dir = Path(f"results/autocorrelation/synthetic/{method}/global")
        results = Parallel(n_jobs=N_JOBS, verbose=10)(
            delayed(_process_graph)(shape, size, method, c_run, intensity, gdf, n_runs, rhos)
            for c_run, intensity in jobs
        )

        global_results = [row for r in results for row in r]
        global_df = pd.DataFrame(global_results)
        global_df.to_parquet(global_dir / f"{shape}_{size}.parquet")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    5.1s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:    5.4s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    6.9s
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:   10.0s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed:   14.2s
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:   16.8s
[Parallel(n_jobs=-1)]: Done 171 out of 190 | elapsed:   21.1s remaining:    2.3s
[Parallel(n_jobs=-1)]: Done 190 out of 190 | elapsed:   22.5s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_

In [8]:
df = pd.read_parquet("results/autocorrelation/synthetic/random/global/hex_25.parquet")

In [9]:
df

,shape,size,corruption_method,rho,missing_edges,corruption_run,data_run,moran_i,moran_p,moran_z
0,hex,25,random,-0.9,2,0,0,-0.255196,0.035,-1.651570
1,hex,25,random,-0.8,2,0,0,-0.057038,0.474,-0.103919
2,hex,25,random,-0.7,2,0,0,-0.327843,0.006,-2.207730
3,hex,25,random,-0.6,2,0,0,-0.158075,0.213,-0.848322
4,hex,25,random,-0.5,2,0,0,-0.205180,0.087,-1.325454
...,...,...,...,...,...,...,...,...,...,...
36095,hex,25,random,0.5,41,9,9,0.458147,0.031,1.913554
36096,hex,25,random,0.6,41,9,9,0.225888,0.155,1.057105
36097,hex,25,random,0.7,41,9,9,0.677708,0.004,2.775419
36098,hex,25,random,0.8,41,9,9,0.456276,0.030,1.915187
